In [57]:
import pandas as pd 
import numpy as np 
import torch
import torch.nn as nn
import torch.optim as optim 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import classification_report 
from torch.utils.data import DataLoader, TensorDataset 


In [85]:
np.random.seed(10)
samples = 1_024
df = pd.DataFrame({
    'a': np.random.randn(samples),
    'b': np.random.randn(samples),
    'c': np.random.randn(samples),
    'd': np.random.choice(['x', 'y', 'z', 'N/A'], samples),
})

def gen_click(row):
    val = (row['a'] * row['b'] * row['c'] + (1.0 if row['d'] == 'x' else 0.5 if row['d'] == 'y' else -0.5 if row['d'] == 'z' else -1.0))
    return val > 0.0

df['click'] = df.apply(gen_click, axis=1)


X = df.drop(columns=['click'])
Y = df[['click']]

# stratify sampling
X_train, X_test, y_train, y_test = train_test_split(X, Y, stratify=Y, test_size=0.2, random_state=10)

df.head()

,a,b,c,d,click
0,1.331587,-0.007829,0.042090,z,False
1,0.715279,0.365266,-0.271058,x,True
2,-1.545400,-0.400028,-0.170392,N/A,False
3,-0.008384,1.132525,0.077673,x,True
4,0.621336,-0.098616,0.078692,z,False


In [86]:
X_train.head()
X_train.shape[-1]

4

In [87]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


prep = ColumnTransformer([
    ('num', StandardScaler(), ['a', 'b', 'c']),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), ['d'])
])

X_train_tensor = torch.tensor(pd.DataFrame(prep.fit_transform(X_train)).values.astype(np.float32))
y_train_tensor = torch.tensor(y_train.values.astype(np.float32))
X_test_tensor = torch.tensor(pd.DataFrame(prep.transform(X_test)).values.astype(np.float32))
y_test_tensor = torch.tensor(y_test.values.astype(np.float32))

# DataLoader
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=128)

In [88]:
X_train_tensor.shape

torch.Size([819, 6])

In [91]:
class Model(nn.Module):
    def __init__(self, inputSize, outputSize, hiddenSize, dropout=0.2) -> None:
        super(Model, self).__init__()
        self.layers = nn.Sequential(*[
            nn.Linear(inputSize, hiddenSize),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hiddenSize, hiddenSize),
            nn.ReLU(),
            nn.Linear(hiddenSize, outputSize),
            nn.Sigmoid()
        ])
        
    def forward(self, x):
        return self.layers(x)


def train(model, train_loader, epochs=1, lr=0.01):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(device)
    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for i, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            # print(f'step: {i}, {loss.item():.5f}')
        if epoch % 10 == 9:
            print(f'epoch: {epoch+1}, {train_loss / len(train_loader):.5f}')
    return model

def eval(model, test_loader):
    from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score, precision_recall_curve, auc
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    preds, probs, targets = [], [], []
    with torch.no_grad():
        for x, y in test_loader:
            output = model(x.to(device))
            preds.extend((output > 0.5).float().cpu().numpy())
            probs.extend(output.cpu().numpy())
            targets.extend(y.cpu().numpy())
    def flat(x): return np.array(x).flatten()
    preds, probs, targets = flat(preds), flat(probs), flat(targets)
    precision, recall, _ = precision_recall_curve(targets, probs)
    return {
        'accuracy': accuracy_score(targets, preds),
        'f1': f1_score(targets, preds),
        'precision': precision_score(targets, preds),
        'recall': recall_score(targets, preds),
        'roc_auc': roc_auc_score(targets, probs),
        'pr_auc': auc(recall, precision),
    }
    
    
def pred(model, X, prep, threshold):
    X = prep.transform_fit(X)
        
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    
    with torch.no_grad():
        outputs = model(X_tensor.to(device))
        predictions = (outputs > 0.5).float().cpu().numpy()
        probabilities = outputs.cpu().numpy()
    
    return predictions, probabilities

In [94]:
model = Model(inputSize=X_train_tensor.shape[-1], outputSize=1, hiddenSize=16)
model = train(model, train_loader, epochs=10)
eval(model, test_loader)

cuda
epoch: 10, 0.30441
